In [33]:
import os
import pandas as pd

# pred_file_paths = ["gbcmen_lrwomen.csv", "Gradien boosting.csv"]
pred_file_paths = ["Gradient boosting.csv"]
base_path = "c:/nextcloud/Studia - PW/semestr 6/projekt interdyscyplinarny/march-ml-madness-2025/src/mikematuszy/submitted_pred/"
m_teams = pd.read_csv(os.path.join(base_path, "MTeams.csv"))
w_teams = pd.read_csv(os.path.join(base_path, "WTeams.csv"))
m_teams["TeamID"] = m_teams["TeamID"].astype(int)
w_teams["TeamID"] = w_teams["TeamID"].astype(int)
m_teams.drop(columns=["FirstD1Season", "LastD1Season"], inplace=True)
# w_teams.drop(columns=["FirstD1Season", "LastD1Season"], inplace=True)
men_and_women_teams = pd.concat([m_teams, w_teams])

team_ids_in_tournament = [
    1181, 1104, 1458, 1112, 1332, 1140, 1388, 1280,
    1124, 1435, 1433, 1251, 1103, 1285, 1352, 1110,
    1291, 1222, 1397, 1246, 1345, 1155, 1228, 1417,
    1211, 1208, 1429, 1400, 1462, 1270, 1219, 1407,
    1459, 1188, 1120, 1277, 1235, 1401, 1276, 1279,
    1266, 1257, 1166, 1307, 1314, 1361, 1471, 1463,
    1252, 1136, 1106, 1384, 1196, 1385, 1403, 1268,
    1272, 1281, 1242, 1163, 1328, 1116, 1179, 1161,
    1213, 1423, 1303, 1313, 
    3376, 3181, 3314, 3268, 3104, 3452, 3435, 3428,
    3231, 3332, 3162, 3449, 3453, 3313, 3333, 3250,
    3399, 3400, 3395, 3323, 3326, 3397, 3276, 3257,
    3228, 3166, 3304, 3235, 3343, 3378, 3286, 3372,
    3192, 3219, 3456, 3417, 3301, 3261, 3124, 3279,
    3199, 3277, 3350, 3210, 3217, 3206, 3123, 3213,
    3361, 3436, 3380, 3471, 3425, 3163, 3328, 3246,
    3243, 3234, 3329, 3143, 3280, 3355, 3293, 3193,
    3251, 3195, 3117, 3422
]

In [50]:
m_teams = pd.read_csv(os.path.join(base_path, "MTeams.csv"))
w_teams = pd.read_csv(os.path.join(base_path, "WTeams.csv"))
m_teams.drop(columns=["FirstD1Season", "LastD1Season"], inplace=True)

# Ensure TeamID is an integer
m_teams["TeamID"] = m_teams["TeamID"].astype(int)

print(m_teams.columns)

for pred_file_path in pred_file_paths:
    # Read prediction file
    predictions = pd.read_csv(os.path.join(base_path, pred_file_path))

    # Extract TEAM1 and TEAM2 from ID
    predictions["TEAM1"] = predictions["ID"].str.split("_").str[1].astype(int)
    predictions["TEAM2"] = predictions["ID"].str.split("_").str[2].astype(int)

    print(predictions.columns)

    # Merge TEAM1 with m_teams, adding a prefix "team_1" to the new columns
    predictions = pd.merge(
        predictions,
        men_and_women_teams,
        left_on="TEAM1",
        right_on="TeamID",
        how="left",
        suffixes=("", "_team_1")
    )
    predictions.rename(
        columns={col: f"team_1_{col}" for col in m_teams.columns if col != "TeamID"},
        inplace=True
    )

    # Drop the duplicate "TeamID" column (since we already have TEAM1)
    predictions.drop(columns=["TeamID"], inplace=True)

        # Merge TEAM1 with m_teams, adding a prefix "team_1" to the new columns
    predictions = pd.merge(
        predictions,
        men_and_women_teams,
        left_on="TEAM2",
        right_on="TeamID",
        how="left",
        suffixes=("", "_team_2")
    )
    predictions.rename(
        columns={col: f"team_2_{col}" for col in m_teams.columns if col != "TeamID"},
        inplace=True
    )

    # Drop the duplicate "TeamID" column (since we already have TEAM1)
    predictions.drop(columns=["TeamID"], inplace=True)

    predictions = predictions[
    (predictions["TEAM1"].isin(team_ids_in_tournament)) & 
    (predictions["TEAM2"].isin(team_ids_in_tournament))
    ]
    predictions.drop(predictions.columns[0], axis=1, inplace=True)
    # predictions.drop(predictions.columns[0], axis=1, inplace=True)


Index(['TeamID', 'TeamName'], dtype='object')
Index(['ID', 'Pred', 'TEAM1', 'TEAM2'], dtype='object')


In [53]:
predictions.to_csv("predictions.csv", index=False)


In [52]:
predictions

,Pred,TEAM1,TEAM2,team_1_TeamName,team_2_TeamName
725,0.251731,1103,1104,Akron,Alabama
727,0.601236,1103,1106,Akron,Alabama St
730,0.521279,1103,1110,Akron,American Univ
732,0.406835,1103,1112,Akron,Arizona
736,0.451958,1103,1116,Akron,Arkansas
...,...,...,...,...,...
131004,0.796841,3452,3456,West Virginia,William & Mary
131019,0.796841,3452,3471,West Virginia,UC San Diego
131031,0.626865,3453,3456,WI Green Bay,William & Mary
131046,0.642615,3453,3471,WI Green Bay,UC San Diego
